In [14]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [15]:
def objective(trial):
    """Optuna objective function"""
    try:
        print(f"\n=== Starting Trial {trial.number} ===")
        
        # 강력한 메모리 정리
        clear_gpu_memory()
        
        # 하이퍼파라미터 샘플링 (고정된 값 공간 사용)
        config_dict = {
            'patch_len': trial.suggest_categorical('patch_len', [7, 14]),
            'stride': trial.suggest_categorical('stride', [1, 2, 4]),  # stride 옵션 추가
            'd_model': trial.suggest_categorical('d_model', [64, 128]),  # 고정
            'n_heads': trial.suggest_categorical('n_heads', [4, 8]),
            'n_layers': trial.suggest_categorical('n_layers', [2, 3]),  # 고정
            'dropout': trial.suggest_float('dropout', 0.0, 0.3),
            'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [8, 16] if SMALL_GPU else [16, 32]),
            'use_instancenorm': trial.suggest_categorical('use_instancenorm', [True, False]),
            'loss_type': trial.suggest_categorical('loss_type', ['mixed', 'huber', 'log1p_mse']),
        }
        
        # 패치 파라미터 검증 및 조정
        patch_len = config_dict['patch_len']
        stride = config_dict['stride']
        seq_len = config.input_len  # 28
        
        # stride가 너무 크면 조정
        if stride >= patch_len:
            config_dict['stride'] = max(1, patch_len // 2)
            print(f"Adjusted stride from {stride} to {config_dict['stride']}")
        
        # 최소 패치 수 보장
        min_patches_needed = 2
        max_stride = (seq_len - patch_len) // (min_patches_needed - 1) if min_patches_needed > 1 else seq_len
        if config_dict['stride'] > max_stride and max_stride > 0:
            config_dict['stride'] = max(1, max_stride)
            print(f"Adjusted stride to {config_dict['stride']} for minimum patches")
        
        print(f"Using patch_len={config_dict['patch_len']}, stride={config_dict['stride']}, seq_len={seq_len}")
        
        # 손실별 추가 파라미터
        if config_dict['loss_type'] == 'mixed':
            config_dict['alpha'] = trial.suggest_float('alpha', 0.6, 0.9)
        elif config_dict['loss_type'] == 'huber':
            config_dict['huber_delta'] = trial.suggest_float('huber_delta', 0.5, 2.0)
        
        # Prior 손실 파라미터
        config_dict['lambda_zero'] = trial.suggest_float('lambda_zero', 0.0, 0.5)
        config_dict['mu_recon'] = trial.suggest_float('mu_recon', 0.0, 0.3)
        config_dict['c_prior'] = trial.suggest_float('c_prior', 0.1, 0.5)
        
        # 데이터 로딩 및 전처리
        print(f"Loading and processing data for trial {trial.number}...")
        train_df, _ = load_data()
        
        # 데이터 타입 명시적 변환
        train_df['sales'] = pd.to_numeric(train_df['sales'], errors='coerce').astype(np.float32)
        train_df = train_df.dropna(subset=['sales'])  # NaN 제거
        
        train_df = engineer_features(train_df)
        
        # 데이터 크기 축소 (메모리 절약)
        unique_store_menus = train_df['store_menu'].unique()
        selected_store_menus = unique_store_menus[:min(len(unique_store_menus), 10)]  # 10개로 더 줄임
        train_df_subset = train_df[train_df['store_menu'].isin(selected_store_menus)]
        
        print(f"Creating sequences for {len(selected_store_menus)} store_menus...")
        sequences = create_sequences(train_df_subset, config.input_len, config.pred_len, config.gap)
        
        if len(sequences) < 50:  # 더 관대한 기준
            print(f"Trial {trial.number}: Not enough sequences ({len(sequences)})")
            return float('inf')
        
        # 시퀀스 수도 제한
        sequences = sequences[:min(len(sequences), 200)]  # 200개로 줄임
        print(f"Using {len(sequences)} sequences")
        
        # 데이터 검증
        for i, seq in enumerate(sequences):
            if np.any(np.isnan(seq['input_features'])) or np.any(np.isnan(seq['target'])):
                print(f"Warning: NaN found in sequence {i}")
                sequences[i]['input_features'] = np.nan_to_num(seq['input_features'], nan=0.0)
                sequences[i]['target'] = np.nan_to_num(seq['target'], nan=0.0)
        
        # 매장 인코더
        stores = list(set([seq['store'] for seq in sequences]))
        store_encoder = LabelEncoder()
        store_encoder.fit(stores)
        
        # Cross validation
        print(f"Starting CV for trial {trial.number}...")
        fold_results, overall_smape = blocked_rolling_cv(sequences, store_encoder, config_dict, config.n_folds, config.gap)
        
        if not fold_results:
            print(f"Trial {trial.number}: No fold results")
            return float('inf')
        
        print(f"Trial {trial.number}: CV completed with sMAPE {overall_smape:.4f}")
        
        # Trial 결과 저장 (성공한 경우만)
        if overall_smape != float('inf'):
            trial_id = trial.number
            trial_dir = os.path.join(config.trials_dir, f'trial_{trial_id:03d}')
            os.makedirs(trial_dir, exist_ok=True)
            
            # 설정 저장
            with open(os.path.join(trial_dir, 'config.json'), 'w') as f:
                json.dump(config_dict, f, indent=2)
            
            # 최고 성능 모델 저장
            best_fold = min(fold_results, key=lambda x: x['smape'])
            torch.save(best_fold['model_state'], os.path.join(trial_dir, 'best.pt'))
            
            # 검증 결과 저장
            val_metrics = {
                'overall_smape': overall_smape,
                'fold_smapes': [r['smape'] for r in fold_results],
            }
            
            with open(os.path.join(trial_dir, 'val_metrics.json'), 'w') as f:
                json.dump(val_metrics, f, indent=2)
            
            # 예측 결과 저장
            fold_predictions_df = pd.DataFrame({
                'fold': [r['fold'] for r in fold_results for _ in range(len(r['predictions']))],
                'store_menu': [sm for r in fold_results for sm in r['store_menus']],
                'prediction': [pred for r in fold_results for pred in r['predictions'].flatten()],
                'target': [target for r in fold_results for target in r['targets'].flatten()]
            })
            fold_predictions_df.to_parquet(os.path.join(trial_dir, 'fold_preds.parquet'))
        
        print(f"Trial {trial_id} completed - sMAPE: {overall_smape:.4f}")
        
        # GPU 메모리 정리
        clear_gpu_memory()
        
        return overall_smape
        
    except torch.cuda.OutOfMemoryError as e:
        print(f"Trial {trial.number} failed: GPU out of memory - {str(e)}")
        clear_gpu_memory()
        return float('inf')
    except Exception as e:
        print(f"Trial {trial.number} failed: {str(e)}")
        clear_gpu_memory()
        return float('inf')
    finally:
        # 최종 메모리 정리
        clear_gpu_memory()
        print(f"=== Trial {trial.number} finished ===\n")# 1. 패키지 임포트 및 환경 설정
import os
import json
import warnings
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import optuna
from optuna.storages import RDBStorage
import logging

warnings.filterwarnings('ignore')

# GPU 사용 설정 (메모리 최적화)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    
    # GPU 메모리 정리
    torch.cuda.empty_cache()
    
    # GPU 메모리 확인
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_memory:.1f} GB")
    
    device = torch.device('cuda')
    # GPU 최적화 설정
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    PIN_MEMORY = True
    
    # GPU 메모리 부족 방지 설정
    if gpu_memory < 8.0:  # 8GB 미만이면 더 보수적
        print("Small GPU memory detected, using conservative settings")
        SMALL_GPU = True
    else:
        SMALL_GPU = False
else:
    print("CUDA not available, using CPU")
    device = torch.device('cpu')
    PIN_MEMORY = False
    SMALL_GPU = False

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# 2. 설정 및 하이퍼파라미터
class Config:
    def __init__(self):
        # 데이터 경로
        self.train_path = './dataset/train.csv'
        self.test_path_pattern = './dataset/TEST_{:02d}.csv'
        self.submission_path = './result/sample_submission.csv'
        
        # 출력 경로
        self.artifacts_dir = './artifacts_patchtst'
        self.trials_dir = os.path.join(self.artifacts_dir, 'trials')
        self.best_dir = os.path.join(self.artifacts_dir, 'best')
        self.result_dir = './result'
        
        # 윈도우 설정
        self.input_len = 28
        self.pred_len = 7
        self.gap = 7
        
        # Optuna 설정
        self.n_trials = 50
        self.storage_url = f'sqlite:///{self.artifacts_dir}/optuna.sqlite3'
        self.study_name = 'patchtst_optimization'
        
        # 검증 설정
        self.n_folds = 4
        self.top_k = 5
        
        # 기본 하이퍼파라미터 (GPU 메모리에 따라 조정)
        if SMALL_GPU:
            self.batch_size = 8
            self.max_epochs = 20
        else:
            self.batch_size = 16
            self.max_epochs = 30
            
        self.patience = 8
        self.grad_clip = 1.0
        self.use_amp = torch.cuda.is_available()  # GPU에서만 AMP 사용
        
        # 앙상블 방법
        self.ensemble_method = 'weighted'  # ['mean', 'weighted', 'median', 'rank_avg']
        
        # 디렉토리 생성
        for dir_path in [self.artifacts_dir, self.trials_dir, self.best_dir, self.result_dir]:
            os.makedirs(dir_path, exist_ok=True)

config = Config()

# 3. 매장별 도메인 지식 (STORE_PRIOR)
STORE_PRIOR = {
    "화담숲주막": {
        "closed_month_range": [(11, 3)]  # 11월말~3월말 휴무
    },
    "화담숲카페": {
        "closed_month_range": [(11, 3)]  # 11월말~3월말 휴무
    }
}

# 4. 데이터 로딩 및 전처리
def load_data():
    """데이터 로딩"""
    train_df = pd.read_csv(config.train_path)
    train_df['date'] = pd.to_datetime(train_df['date'])
    train_df = train_df.sort_values(['store_menu', 'date']).reset_index(drop=True)
    
    # 테스트 데이터 로딩
    test_dfs = []
    for i in range(10):
        test_path = config.test_path_pattern.format(i)
        if os.path.exists(test_path):
            test_df = pd.read_csv(test_path)
            test_df['date'] = pd.to_datetime(test_df['date'])
            test_df['test_id'] = i
            test_dfs.append(test_df)
    
    return train_df, test_dfs

def get_prior_features(store, dates):
    """매장별 도메인 특징 계산"""
    prior = STORE_PRIOR.get(store, {})
    n_days = len(dates)
    
    # 기본값 초기화 (모든 매장 기본적으로 영업)
    prior_open_prob = np.ones(n_days, dtype=np.float32)
    
    # 휴무 기간 처리 (화담숲주막, 화담숲카페만)
    closed_ranges = prior.get("closed_month_range", [])
    for i, date in enumerate(dates):
        month = date.month
        
        for start_month, end_month in closed_ranges:
            if start_month > end_month:  # 연도를 넘나드는 경우 (11월~3월)
                if month >= start_month or month <= end_month:
                    prior_open_prob[i] = 0.0  # 완전 휴무
            else:
                if start_month <= month <= end_month:
                    prior_open_prob[i] = 0.0  # 완전 휴무
    
    return {
        'prior_open_prob': prior_open_prob
    }

def engineer_features(df):
    """특징 공학"""
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    # 캘린더 특징
    df['dow'] = df['date'].dt.dayofweek.astype(np.float32)
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(np.float32)
    df['month'] = df['date'].dt.month.astype(np.float32)
    
    # sin/cos 인코딩
    df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7).astype(np.float32)
    df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7).astype(np.float32)
    df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52).astype(np.float32)
    df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52).astype(np.float32)
    
    # sales 컬럼도 float32로 변환
    df['sales'] = df['sales'].astype(np.float32)
    
    # 매장별 집계
    df = df.sort_values(['store_menu', 'date']).reset_index(drop=True)
    
    # 이동 통계 (그룹별) - 모든 결과를 float32로 변환
    for window in [7, 14, 28]:
        df[f'ma_{window}'] = df.groupby('store_menu')['sales'].transform(
            lambda x: x.rolling(window, min_periods=1).mean().astype(np.float32)
        )
        df[f'std_{window}'] = df.groupby('store_menu')['sales'].transform(
            lambda x: x.rolling(window, min_periods=1).std().fillna(0).astype(np.float32)
        )
        df[f'min_{window}'] = df.groupby('store_menu')['sales'].transform(
            lambda x: x.rolling(window, min_periods=1).min().astype(np.float32)
        )
        df[f'max_{window}'] = df.groupby('store_menu')['sales'].transform(
            lambda x: x.rolling(window, min_periods=1).max().astype(np.float32)
        )
    
    return df

def create_sequences(df, input_len=28, pred_len=7, gap=7):
    """시계열 시퀀스 생성"""
    sequences = []
    
    for store_menu in df['store_menu'].unique():
        store_df = df[df['store_menu'] == store_menu].copy()
        store_df = store_df.sort_values('date').reset_index(drop=True)
        
        if len(store_df) < input_len + gap + pred_len:
            continue
            
        store = store_df['store'].iloc[0]
        
        for i in range(len(store_df) - input_len - gap - pred_len + 1):
            # 입력 시퀀스
            input_seq = store_df.iloc[i:i+input_len].copy()
            
            # 예측 대상
            target_start = i + input_len + gap
            target_seq = store_df.iloc[target_start:target_start+pred_len].copy()
            
            # Prior 특징 계산
            input_dates = pd.to_datetime(input_seq['date'])
            prior_features = get_prior_features(store, input_dates)
            
            # 매장 총합 특징
            store_total = store_df.groupby('date')['sales'].sum().reset_index()
            store_total_dict = dict(zip(store_total['date'], store_total['sales']))
            store_total_seq = [store_total_dict.get(date, 0.0) for date in input_dates]
            
            # Trend slope (28일 선형회귀 기울기)
            y_vals = input_seq['sales'].values.astype(np.float32)
            x_vals = np.arange(len(y_vals), dtype=np.float32)
            if len(y_vals) > 1:
                slope = np.polyfit(x_vals, y_vals, 1)[0]
            else:
                slope = 0.0
            slope = float(slope)  # float32로 변환
            
            # 특징 벡터 구성
            feature_cols = ['sales', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos'] + \
                          [f'ma_{w}' for w in [7, 14, 28]] + \
                          [f'std_{w}' for w in [7, 14, 28]] + \
                          [f'min_{w}' for w in [7, 14, 28]] + \
                          [f'max_{w}' for w in [7, 14, 28]]
            
            input_features = input_seq[feature_cols].values.astype(np.float32)
            
            # Prior 특징 추가 (prior_open_prob만)
            prior_features_arr = np.column_stack([
                prior_features['prior_open_prob'].astype(np.float32),
                np.full(input_len, slope, dtype=np.float32),
                np.array(store_total_seq, dtype=np.float32)
            ])
            
            input_features = np.concatenate([input_features, prior_features_arr], axis=1).astype(np.float32)
            target_values = target_seq['sales'].values.astype(np.float32)
            
            # NaN 값 체크 및 제거
            if np.any(np.isnan(input_features)) or np.any(np.isnan(target_values)):
                continue
                
            sequences.append({
                'store_menu': store_menu,
                'store': store,
                'input_features': input_features,
                'target': target_values,
                'input_dates': input_dates.tolist(),
                'target_dates': pd.to_datetime(target_seq['date']).tolist()
            })
    
    return sequences

# 5. PatchTST 모델 구현
class PatchTST(nn.Module):
    def __init__(self, configs):
        super(PatchTST, self).__init__()
        self.seq_len = configs['seq_len']
        self.pred_len = configs['pred_len']
        self.patch_len = configs['patch_len']
        self.stride = configs['stride']
        self.d_model = configs['d_model']
        self.n_heads = configs['n_heads']
        self.n_layers = configs['n_layers']
        self.dropout = configs['dropout']
        self.enc_in = configs['enc_in']
        self.use_instancenorm = configs.get('use_instancenorm', True)
        self.n_stores = configs.get('n_stores', 10)
        
        # 패치 수 계산 수정
        self.patch_num = max(1, (self.seq_len - self.patch_len) // self.stride + 1)
        print(f"PatchTST init: seq_len={self.seq_len}, patch_len={self.patch_len}, stride={self.stride}, patch_num={self.patch_num}")
        
        # Instance Normalization
        if self.use_instancenorm:
            self.instance_norm = nn.InstanceNorm1d(self.enc_in)
        
        # 패치 임베딩
        self.patch_embedding = nn.Linear(self.patch_len, self.d_model)
        
        # 위치 임베딩
        self.positional_encoding = nn.Parameter(torch.randn(self.patch_num, self.d_model))
        
        # 매장 임베딩
        self.store_embedding = nn.Embedding(self.n_stores, self.d_model)
        
        # Transformer 인코더
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.n_heads,
            dropout=self.dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.n_layers)
        
        # 출력 프로젝션
        self.head = nn.Linear(self.d_model, self.pred_len)
        
        # Store-total 헤드 (부가 손실용)
        self.store_total_head = nn.Linear(self.d_model, self.pred_len)
        
        # FiLM 레이어
        self.film_gamma = nn.Linear(self.d_model, self.d_model)
        self.film_beta = nn.Linear(self.d_model, self.d_model)
        
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, x, store_ids):
        # x: [batch_size, seq_len, enc_in]
        batch_size, seq_len, enc_in = x.shape
        
        # 시퀀스 길이 검증
        if seq_len != self.seq_len:
            print(f"Warning: Expected seq_len={self.seq_len}, got {seq_len}")
            
        # Instance Normalization
        if self.use_instancenorm:
            x = x.transpose(1, 2)  # [batch_size, enc_in, seq_len]
            x = self.instance_norm(x)
            x = x.transpose(1, 2)  # [batch_size, seq_len, enc_in]
        
        # 채널별 독립 처리
        outputs = []
        store_outputs = []
        
        for i in range(enc_in):
            channel_data = x[:, :, i]  # [batch_size, seq_len]
            
            # 패치 생성 (안전한 인덱싱)
            patches = []
            actual_patch_num = 0
            
            for j in range(self.patch_num):
                start_idx = j * self.stride
                end_idx = start_idx + self.patch_len
                
                if end_idx <= seq_len:
                    patch = channel_data[:, start_idx:end_idx]  # [batch_size, patch_len]
                    patches.append(patch)
                    actual_patch_num += 1
                else:
                    # 시퀀스가 부족한 경우 패딩
                    available_len = seq_len - start_idx
                    if available_len > 0:
                        patch = channel_data[:, start_idx:]  # [batch_size, available_len]
                        # 제로 패딩
                        padding = torch.zeros(batch_size, self.patch_len - available_len, device=patch.device)
                        patch = torch.cat([patch, padding], dim=1)
                        patches.append(patch)
                        actual_patch_num += 1
                    break
            
            if not patches:
                # 패치가 없는 경우 전체 시퀀스를 하나의 패치로 사용
                if seq_len >= self.patch_len:
                    patch = channel_data[:, :self.patch_len]
                else:
                    patch = channel_data
                    padding = torch.zeros(batch_size, self.patch_len - seq_len, device=patch.device)
                    patch = torch.cat([patch, padding], dim=1)
                patches = [patch]
                actual_patch_num = 1
            
            patches = torch.stack(patches, dim=1)  # [batch_size, actual_patch_num, patch_len]
            
            # 패치 임베딩
            patch_emb = self.patch_embedding(patches)  # [batch_size, actual_patch_num, d_model]
            
            # 위치 임베딩 추가 (크기 맞추기)
            pos_emb = self.positional_encoding[:actual_patch_num].unsqueeze(0)  # [1, actual_patch_num, d_model]
            patch_emb = patch_emb + pos_emb
            
            # 매장 임베딩 및 FiLM
            store_emb = self.store_embedding(store_ids)  # [batch_size, d_model]
            gamma = self.film_gamma(store_emb).unsqueeze(1)  # [batch_size, 1, d_model]
            beta = self.film_beta(store_emb).unsqueeze(1)   # [batch_size, 1, d_model]
            
            patch_emb = gamma * patch_emb + beta
            
            # Transformer
            patch_emb = self.dropout_layer(patch_emb)
            encoded = self.transformer(patch_emb)  # [batch_size, actual_patch_num, d_model]
            
            # Global average pooling
            pooled = encoded.mean(dim=1)  # [batch_size, d_model]
            
            # 예측
            channel_output = self.head(pooled)  # [batch_size, pred_len]
            outputs.append(channel_output)
            
            # Store-total 헤드 (첫 번째 채널에서만)
            if i == 0:
                store_total_output = self.store_total_head(pooled)
                store_outputs.append(store_total_output)
        
        # 채널별 출력을 합산
        output = torch.stack(outputs, dim=-1).sum(dim=-1)  # [batch_size, pred_len]
        
        # 음수 클리핑
        output = torch.clamp(output, min=0)
        
        store_total_output = store_outputs[0] if store_outputs else None
        
        return output, store_total_output

# 6. 손실 함수
class MixedLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super().__init__()
        self.alpha = alpha
        
    def log1p_mse(self, pred, target):
        return F.mse_loss(torch.log1p(pred), torch.log1p(target))
    
    def smape_surrogate(self, pred, target, eps=1e-8):
        numerator = torch.abs(pred - target)
        denominator = torch.abs(pred) + torch.abs(target) + eps
        return (2 * numerator / denominator).mean()
    
    def forward(self, pred, target):
        log1p_loss = self.log1p_mse(pred, target)
        smape_loss = self.smape_surrogate(pred, target)
        return self.alpha * log1p_loss + (1 - self.alpha) * smape_loss

class HuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta
        
    def forward(self, pred, target):
        return F.huber_loss(pred, target, delta=self.delta)

class PriorAwareLoss(nn.Module):
    def __init__(self, lambda_zero=0.1, mu_recon=0.1, c_prior=0.3):
        super().__init__()
        self.lambda_zero = lambda_zero
        self.mu_recon = mu_recon
        self.c_prior = c_prior
        
    def zero_inflation_penalty(self, pred, prior_open_prob):
        threshold = self.c_prior * prior_open_prob
        penalty = F.relu(pred - threshold.unsqueeze(-1))
        return (penalty ** 2).mean()
    
    def store_recon_loss(self, pred, store_total_pred):
        if store_total_pred is not None:
            pred_sum = pred.sum(dim=0)  # sum over batch (same store)
            return F.l1_loss(pred_sum, store_total_pred.sum(dim=0))
        return 0
    
    def forward(self, pred, target, prior_open_prob, store_total_pred=None):
        zero_penalty = self.zero_inflation_penalty(pred, prior_open_prob)
        recon_loss = self.store_recon_loss(pred, store_total_pred)
        
        return self.lambda_zero * zero_penalty + self.mu_recon * recon_loss

class TimeSeriesDataset(Dataset):
    def __init__(self, sequences, store_encoder):
        self.sequences = sequences
        self.store_encoder = store_encoder
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        
        input_features = torch.FloatTensor(seq['input_features'].astype(np.float32))
        target = torch.FloatTensor(seq['target'].astype(np.float32))
        store_id = self.store_encoder.transform([seq['store']])[0]
        
        # Prior 특징 추출 (prior_open_prob 채널만)
        prior_open_prob = input_features[:, -3].clone()  # prior_open_prob 채널
        
        return {
            'input_features': input_features,
            'target': target,
            'store_id': torch.LongTensor([store_id]),
            'prior_open_prob': prior_open_prob,
            'store_menu': seq['store_menu']
        }

# 8. 학습 및 검증 함수
def smape_metric(pred, target, eps=1e-8):
    """sMAPE 계산"""
    pred = np.clip(pred, 0, None)  # 음수 클리핑
    numerator = np.abs(pred - target)
    denominator = np.abs(pred) + np.abs(target) + eps
    return 200 * np.mean(numerator / denominator)

def train_model(model, train_loader, val_loader, config_dict, device):
    """모델 학습"""
    print(f"Training on device: {device}")
    if torch.cuda.is_available():
        print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    
    model = model.to(device)
    
    # 메모리 최적화
    clear_gpu_memory()
    if torch.cuda.is_available():
        print(f"Model moved to GPU. Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    
    # 손실 함수 설정
    if config_dict['loss_type'] == 'mixed':
        criterion = MixedLoss(alpha=config_dict.get('alpha', 0.7))
    elif config_dict['loss_type'] == 'huber':
        criterion = HuberLoss(delta=config_dict.get('huber_delta', 1.0))
    else:  # log1p_mse
        criterion = nn.MSELoss()
    
    prior_loss = PriorAwareLoss(
        lambda_zero=config_dict.get('lambda_zero', 0.0),
        mu_recon=config_dict.get('mu_recon', 0.0),
        c_prior=config_dict.get('c_prior', 0.3)
    )
    
    optimizer = AdamW(model.parameters(), 
                      lr=config_dict['lr'], 
                      weight_decay=config_dict['weight_decay'])
    
    scheduler = CosineAnnealingLR(optimizer, T_max=config.max_epochs)
    
    scaler = torch.cuda.amp.GradScaler() if (config.use_amp and torch.cuda.is_available()) else None
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    train_losses = []
    val_losses = []
    
    for epoch in range(config.max_epochs):
        # Training
        model.train()
        total_train_loss = 0
        successful_batches = 0
        
        for batch_idx, batch in enumerate(train_loader):
            try:
                optimizer.zero_grad()
                
                input_features = batch['input_features'].to(device, non_blocking=True)
                target = batch['target'].to(device, non_blocking=True)
                store_ids = batch['store_id'].squeeze().to(device, non_blocking=True)
                prior_open_prob = batch['prior_open_prob'].to(device, non_blocking=True)
                
                if config.use_amp and torch.cuda.is_available():
                    with torch.cuda.amp.autocast():
                        pred, store_total_pred = model(input_features, store_ids)
                        
                        if config_dict['loss_type'] == 'log1p_mse':
                            main_loss = F.mse_loss(torch.log1p(pred), torch.log1p(target))
                        else:
                            main_loss = criterion(pred, target)
                        
                        prior_loss_val = prior_loss(pred, target, prior_open_prob, store_total_pred)
                        total_loss = main_loss + prior_loss_val
                    
                    scaler.scale(total_loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    pred, store_total_pred = model(input_features, store_ids)
                    
                    if config_dict['loss_type'] == 'log1p_mse':
                        main_loss = F.mse_loss(torch.log1p(pred), torch.log1p(target))
                    else:
                        main_loss = criterion(pred, target)
                    
                    prior_loss_val = prior_loss(pred, target, prior_open_prob, store_total_pred)
                    total_loss = main_loss + prior_loss_val
                    
                    total_loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                    optimizer.step()
                
                total_train_loss += total_loss.item()
                successful_batches += 1
                
                # GPU 메모리 관리
                if batch_idx % 5 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    
            except torch.cuda.OutOfMemoryError:
                print(f"GPU OOM at batch {batch_idx}, skipping...")
                clear_gpu_memory()
                continue
            except Exception as e:
                print(f"Error at batch {batch_idx}: {str(e)}")
                continue
        
        if successful_batches == 0:
            print("No successful batches in this epoch!")
            break
        
        # Validation
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_targets = []
        successful_val_batches = 0
        
        try:
            with torch.no_grad():
                for batch_idx, batch in enumerate(val_loader):
                    try:
                        input_features = batch['input_features'].to(device, non_blocking=True)
                        target = batch['target'].to(device, non_blocking=True)
                        store_ids = batch['store_id'].squeeze().to(device, non_blocking=True)
                        prior_open_prob = batch['prior_open_prob'].to(device, non_blocking=True)
                        
                        pred, store_total_pred = model(input_features, store_ids)
                        
                        if config_dict['loss_type'] == 'log1p_mse':
                            main_loss = F.mse_loss(torch.log1p(pred), torch.log1p(target))
                        else:
                            main_loss = criterion(pred, target)
                        
                        prior_loss_val = prior_loss(pred, target, prior_open_prob, store_total_pred)
                        total_loss = main_loss + prior_loss_val
                        
                        total_val_loss += total_loss.item()
                        successful_val_batches += 1
                        
                        val_preds.append(pred.cpu().numpy())
                        val_targets.append(target.cpu().numpy())
                        
                    except torch.cuda.OutOfMemoryError:
                        print(f"GPU OOM at val batch {batch_idx}, skipping...")
                        clear_gpu_memory()
                        continue
                        
        except Exception as e:
            print(f"Validation error: {str(e)}")
            val_preds = [np.zeros((1, config.pred_len))]
            val_targets = [np.zeros((1, config.pred_len))]
            total_val_loss = float('inf')
            successful_val_batches = 1
        
        avg_train_loss = total_train_loss / max(successful_batches, 1)
        avg_val_loss = total_val_loss / max(successful_val_batches, 1)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        # sMAPE 계산
        try:
            val_preds_np = np.concatenate(val_preds, axis=0)
            val_targets_np = np.concatenate(val_targets, axis=0)
            val_smape = smape_metric(val_preds_np, val_targets_np)
        except:
            val_smape = 999.0
        
        scheduler.step()
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            try:
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            except:
                best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if epoch % 3 == 0:  # 더 자주 출력
            print(f"Epoch {epoch}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val sMAPE: {val_smape:.4f}")
            if torch.cuda.is_available():
                print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        
        if patience_counter >= config.patience:
            print(f"Early stopping at epoch {epoch}")
            break
            
        # 에폭마다 메모리 정리
        if epoch % 2 == 0:
            clear_gpu_memory()
    
    # 최적 모델 복원
    if best_model_state is not None:
        try:
            model.load_state_dict(best_model_state)
        except:
            print("Failed to load best model state")
    
    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'final_val_smape': val_smape if 'val_smape' in locals() else 999.0
    }

def clear_gpu_memory():
    """GPU 메모리 완전 정리"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    # 가비지 컬렉션 강제 실행
    import gc
    gc.collect()

def blocked_rolling_cv(sequences, store_encoder, config_dict, n_folds=4, gap=7):
    """Blocked Rolling-Origin Cross Validation"""
    # 시간순 정렬
    sequences_sorted = sorted(sequences, key=lambda x: x['input_dates'][-1])
    
    n_samples = len(sequences_sorted)
    fold_size = n_samples // n_folds
    
    fold_results = []
    all_predictions = []
    
    for fold in range(n_folds):
        print(f"\n--- Fold {fold + 1}/{n_folds} ---")
        
        # 메모리 정리
        clear_gpu_memory()
        
        # Train/Val 분할 (시간 기준)
        val_start = fold * fold_size
        val_end = (fold + 1) * fold_size if fold < n_folds - 1 else n_samples
        
        val_sequences = sequences_sorted[val_start:val_end]
        train_sequences = sequences_sorted[:val_start] + sequences_sorted[val_end:]
        
        if not train_sequences or not val_sequences:
            continue
        
        # 데이터셋 생성
        train_dataset = TimeSeriesDataset(train_sequences, store_encoder)
        val_dataset = TimeSeriesDataset(val_sequences, store_encoder)
        
        train_loader = DataLoader(train_dataset, 
                                 batch_size=config_dict['batch_size'], 
                                 shuffle=True, 
                                 pin_memory=PIN_MEMORY,
                                 num_workers=0)  # GPU 사용시 worker 0으로 고정
        val_loader = DataLoader(val_dataset, 
                               batch_size=config_dict['batch_size'], 
                               shuffle=False,
                               pin_memory=PIN_MEMORY,
                               num_workers=0)
        
        # 모델 생성
        model_config = {
            'seq_len': config.input_len,
            'pred_len': config.pred_len,
            'patch_len': config_dict['patch_len'],
            'stride': config_dict['stride'],
            'd_model': config_dict['d_model'],
            'n_heads': config_dict['n_heads'],
            'n_layers': config_dict['n_layers'],
            'dropout': config_dict['dropout'],
            'enc_in': train_sequences[0]['input_features'].shape[1],
            'use_instancenorm': config_dict['use_instancenorm'],
            'n_stores': len(store_encoder.classes_)
        }
        
        model = PatchTST(model_config)
        
        try:
            # 학습
            trained_model, train_metrics = train_model(model, train_loader, val_loader, config_dict, device)
            
            # 검증 예측
            trained_model.eval()
            fold_preds = []
            fold_targets = []
            fold_store_menus = []
            
            with torch.no_grad():
                for batch in val_loader:
                    input_features = batch['input_features'].to(device)
                    target = batch['target']
                    store_ids = batch['store_id'].squeeze().to(device)
                    store_menus = batch['store_menu']
                    
                    pred, _ = trained_model(input_features, store_ids)
                    pred = pred.cpu().numpy()
                    
                    fold_preds.extend(pred)
                    fold_targets.extend(target.numpy())
                    fold_store_menus.extend(store_menus)
            
            fold_preds = np.array(fold_preds)
            fold_targets = np.array(fold_targets)
            
            # sMAPE 계산
            fold_smape = smape_metric(fold_preds, fold_targets)
            
            fold_results.append({
                'fold': fold,
                'smape': fold_smape,
                'predictions': fold_preds,
                'targets': fold_targets,
                'store_menus': fold_store_menus,
                'model_state': trained_model.state_dict().copy(),  # 복사본 저장
                'train_metrics': train_metrics
            })
            
            print(f"Fold {fold + 1} sMAPE: {fold_smape:.4f}")
            
        except Exception as e:
            print(f"Fold {fold + 1} failed: {str(e)}")
            # 실패한 경우에도 메모리 정리
            
        finally:
            # 명시적 메모리 정리
            try:
                del model
                del trained_model
                del train_loader
                del val_loader
                del train_dataset
                del val_dataset
            except:
                pass
            clear_gpu_memory()
    
    if not fold_results:
        return [], float('inf')
    
    # 전체 성능 계산
    all_preds = np.concatenate([r['predictions'] for r in fold_results])
    all_targets = np.concatenate([r['targets'] for r in fold_results])
    overall_smape = smape_metric(all_preds, all_targets)
    
    return fold_results, overall_smape

# 10. Optuna 최적화
def objective(trial):
    """Optuna objective function"""
    try:
        # GPU 메모리 정리
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # 하이퍼파라미터 샘플링 (고정된 값 공간 사용)
        config_dict = {
            'patch_len': trial.suggest_categorical('patch_len', [7, 14]),
            'stride': trial.suggest_categorical('stride', [1, 2]),
            'd_model': trial.suggest_categorical('d_model', [64, 128]),  # 고정
            'n_heads': trial.suggest_categorical('n_heads', [4, 8]),
            'n_layers': trial.suggest_categorical('n_layers', [2, 3]),  # 고정
            'dropout': trial.suggest_float('dropout', 0.0, 0.3),
            'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [16, 32]),  # 고정 (GPU 안전)
            'use_instancenorm': trial.suggest_categorical('use_instancenorm', [True, False]),
            'loss_type': trial.suggest_categorical('loss_type', ['mixed', 'huber', 'log1p_mse']),
        }
        
        # 손실별 추가 파라미터
        if config_dict['loss_type'] == 'mixed':
            config_dict['alpha'] = trial.suggest_float('alpha', 0.6, 0.9)
        elif config_dict['loss_type'] == 'huber':
            config_dict['huber_delta'] = trial.suggest_float('huber_delta', 0.5, 2.0)
        
        # Prior 손실 파라미터
        config_dict['lambda_zero'] = trial.suggest_float('lambda_zero', 0.0, 0.5)
        config_dict['mu_recon'] = trial.suggest_float('mu_recon', 0.0, 0.3)
        config_dict['c_prior'] = trial.suggest_float('c_prior', 0.1, 0.5)
        
        # 데이터 로딩 및 전처리
        train_df, _ = load_data()
        train_df = engineer_features(train_df)
        sequences = create_sequences(train_df, config.input_len, config.pred_len, config.gap)
        
        # 매장 인코더
        stores = list(set([seq['store'] for seq in sequences]))
        store_encoder = LabelEncoder()
        store_encoder.fit(stores)
        
        # Cross validation
        print(f"Starting CV for trial {trial.number}...")
        fold_results, overall_smape = blocked_rolling_cv(sequences, store_encoder, config_dict, config.n_folds, config.gap)
        
        if not fold_results:
            print(f"Trial {trial.number}: No fold results")
            return float('inf')
        
        print(f"Trial {trial.number}: CV completed with sMAPE {overall_smape:.4f}")
        
        # Trial 결과 저장
        trial_id = trial.number
        trial_dir = os.path.join(config.trials_dir, f'trial_{trial_id:03d}')
        os.makedirs(trial_dir, exist_ok=True)
        
        # 설정 저장
        with open(os.path.join(trial_dir, 'config.json'), 'w') as f:
            json.dump(config_dict, f, indent=2)
        
        # 최고 성능 모델 저장
        best_fold = min(fold_results, key=lambda x: x['smape'])
        torch.save(best_fold['model_state'], os.path.join(trial_dir, 'best.pt'))
        
        # 검증 결과 저장
        val_metrics = {
            'overall_smape': overall_smape,
            'fold_smapes': [r['smape'] for r in fold_results],
            'fold_results': fold_results
        }
        
        with open(os.path.join(trial_dir, 'val_metrics.json'), 'w') as f:
            json.dump({
                'overall_smape': overall_smape,
                'fold_smapes': [r['smape'] for r in fold_results]
            }, f, indent=2)
        
        # 예측 결과 저장
        fold_predictions_df = pd.DataFrame({
            'fold': [r['fold'] for r in fold_results for _ in range(len(r['predictions']))],
            'store_menu': [sm for r in fold_results for sm in r['store_menus']],
            'prediction': [pred for r in fold_results for pred in r['predictions'].flatten()],
            'target': [target for r in fold_results for target in r['targets'].flatten()]
        })
        fold_predictions_df.to_parquet(os.path.join(trial_dir, 'fold_preds.parquet'))
        
        print(f"Trial {trial_id} completed - sMAPE: {overall_smape:.4f}")
        
        # GPU 메모리 정리
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return overall_smape
        
    except torch.cuda.OutOfMemoryError as e:
        print(f"Trial {trial.number} failed: GPU out of memory - {str(e)}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float('inf')
    except Exception as e:
        print(f"Trial {trial.number} failed: {str(e)}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float('inf')

def run_optimization():
    """Optuna 최적화 실행"""
    # 기존 완료된 trial 수 확인
    completed_trials = 0
    if os.path.exists(config.trials_dir):
        completed_trials = len([d for d in os.listdir(config.trials_dir) 
                              if os.path.isdir(os.path.join(config.trials_dir, d)) and d.startswith('trial_')])
    
    remaining_trials = max(0, config.n_trials - completed_trials)
    print(f"Completed trials: {completed_trials}, Remaining: {remaining_trials}")
    
    if remaining_trials == 0:
        print("All trials completed!")
        return
    
    # 새로운 study 생성 (기존과 충돌 방지)
    study_name_new = f"{config.study_name}_v2"
    storage = RDBStorage(config.storage_url)
    
    try:
        study = optuna.create_study(
            study_name=study_name_new,
            storage=storage,
            direction='minimize',
            load_if_exists=True
        )
        print(f"Created new study: {study_name_new}")
    except:
        study = optuna.load_study(study_name=study_name_new, storage=storage)
        print(f"Loaded existing study: {study_name_new}")
    
    # 최적화 실행
    study.optimize(objective, n_trials=min(remaining_trials, 10))  # 일단 10개만 시도
    
    print("Optimization completed!")
    if study.trials:
        best_trial = min(study.trials, key=lambda t: t.value if t.value != float('inf') else float('inf'))
        if best_trial.value != float('inf'):
            print(f"Best trial: {best_trial.number}")
            print(f"Best sMAPE: {best_trial.value:.4f}")
            print(f"Best params: {best_trial.params}")
        else:
            print("No successful trials found")
    else:
        print("No trials completed")

# 11. Top-K 모델 선택 및 앙상블
def select_top_models(k=5):
    """상위 k개 모델 선택"""
    trial_results = []
    
    for trial_dir in os.listdir(config.trials_dir):
        trial_path = os.path.join(config.trials_dir, trial_dir)
        if not os.path.isdir(trial_path) or not trial_dir.startswith('trial_'):
            continue
        
        metrics_path = os.path.join(trial_path, 'val_metrics.json')
        if not os.path.exists(metrics_path):
            continue
        
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        
        trial_id = int(trial_dir.split('_')[1])
        trial_results.append({
            'trial_id': trial_id,
            'smape': metrics['overall_smape'],
            'trial_dir': trial_path
        })
    
    # sMAPE 기준 정렬
    trial_results.sort(key=lambda x: x['smape'])
    top_k_trials = trial_results[:k]
    
    print(f"Selected top {k} models:")
    for i, trial in enumerate(top_k_trials):
        print(f"  {i+1}. Trial {trial['trial_id']}: sMAPE = {trial['smape']:.4f}")
    
    return top_k_trials

def ensemble_predictions(predictions_list, method='weighted', smapes=None):
    """앙상블 예측"""
    predictions_array = np.array(predictions_list)  # [n_models, n_samples, pred_len]
    
    if method == 'mean':
        return np.mean(predictions_array, axis=0)
    
    elif method == 'median':
        return np.median(predictions_array, axis=0)
    
    elif method == 'weighted':
        if smapes is None:
            return np.mean(predictions_array, axis=0)
        
        # 1/sMAPE 가중치 (낮은 sMAPE일수록 높은 가중치)
        weights = 1.0 / np.array(smapes)
        weights = weights / weights.sum()
        
        weighted_pred = np.zeros_like(predictions_array[0])
        for i, weight in enumerate(weights):
            weighted_pred += weight * predictions_array[i]
        
        return weighted_pred
    
    elif method == 'rank_avg':
        # 순위 평균
        n_models, n_samples, pred_len = predictions_array.shape
        final_pred = np.zeros((n_samples, pred_len))
        
        for i in range(n_samples):
            for j in range(pred_len):
                values = predictions_array[:, i, j]
                ranks = np.argsort(np.argsort(values))  # 순위 계산
                final_pred[i, j] = np.mean(values[np.argsort(ranks)])
        
        return final_pred
    
    else:
        raise ValueError(f"Unknown ensemble method: {method}")

def generate_ensemble_predictions():
    """앙상블 예측 생성"""
    # Top-K 모델 선택
    top_models = select_top_models(config.top_k)
    
    if not top_models:
        print("No valid models found! Creating dummy predictions...")
        # 더미 예측 생성
        train_df, test_dfs = load_data()
        dummy_predictions = {}
        
        for test_id, test_df in enumerate(test_dfs):
            store_menu_predictions = {}
            for store_menu in test_df['store_menu'].unique():
                # 평균값 기반 더미 예측
                store_menu_data = train_df[train_df['store_menu'] == store_menu]['sales']
                if len(store_menu_data) > 0:
                    mean_val = max(0, store_menu_data.mean())
                    store_menu_predictions[store_menu] = np.full(config.pred_len, mean_val)
                else:
                    store_menu_predictions[store_menu] = np.zeros(config.pred_len)
            
            dummy_predictions[f'TEST_{test_id:02d}'] = store_menu_predictions
        
        return dummy_predictions
    
    # 데이터 로딩
    train_df, test_dfs = load_data()
    train_df = engineer_features(train_df)
    
    # 매장 인코더 (전체 데이터 기준)
    all_stores = train_df['store'].unique()
    store_encoder = LabelEncoder()
    store_encoder.fit(all_stores)
    
    # 각 테스트 파일별 예측
    all_test_predictions = {}
    
    for test_id, test_df in enumerate(test_dfs):
        print(f"\nPredicting TEST_{test_id:02d}...")
        
        test_df = engineer_features(test_df)
        
        # 각 store_menu별 예측
        store_menu_predictions = {}
        
        for store_menu in test_df['store_menu'].unique():
            store_menu_df = test_df[test_df['store_menu'] == store_menu].copy()
            store_menu_df = store_menu_df.sort_values('date').reset_index(drop=True)
            
            if len(store_menu_df) < config.input_len:
                # 데이터가 부족한 경우 0으로 예측
                store_menu_predictions[store_menu] = np.zeros(config.pred_len)
                continue
            
            store = store_menu_df['store'].iloc[0]
            
            # 마지막 28일 사용
            input_seq = store_menu_df.iloc[-config.input_len:].copy()
            input_dates = pd.to_datetime(input_seq['date'])
            
            # Prior 특징 계산
            prior_features = get_prior_features(store, input_dates)
            
            # 매장 총합 특징
            store_total = store_menu_df.groupby('date')['sales'].sum().reset_index()
            store_total_dict = dict(zip(store_total['date'], store_total['sales']))
            store_total_seq = [store_total_dict.get(date, 0) for date in input_dates]
            
            # Trend slope
            y_vals = input_seq['sales'].values
            x_vals = np.arange(len(y_vals))
            slope = np.polyfit(x_vals, y_vals, 1)[0] if len(y_vals) > 1 else 0
            
            # 특징 벡터 구성
            feature_cols = ['sales', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos'] + \
                          [f'ma_{w}' for w in [7, 14, 28]] + \
                          [f'std_{w}' for w in [7, 14, 28]] + \
                          [f'min_{w}' for w in [7, 14, 28]] + \
                          [f'max_{w}' for w in [7, 14, 28]]
            
            input_features = input_seq[feature_cols].values
            
            # Prior 특징 추가 (prior_open_prob만)
            prior_features_arr = np.column_stack([
                prior_features['prior_open_prob'],
                np.full(config.input_len, slope),
                store_total_seq
            ])
            
            input_features = np.concatenate([input_features, prior_features_arr], axis=1)
            
            # 각 모델별 예측
            model_predictions = []
            model_smapes = []
            
            for model_info in top_models:
                try:
                    # 설정 로드
                    config_path = os.path.join(model_info['trial_dir'], 'config.json')
                    with open(config_path, 'r') as f:
                        model_config_dict = json.load(f)
                    
                    # 모델 생성 및 로드
                    model_config = {
                        'seq_len': config.input_len,
                        'pred_len': config.pred_len,
                        'patch_len': model_config_dict['patch_len'],
                        'stride': model_config_dict['stride'],
                        'd_model': model_config_dict['d_model'],
                        'n_heads': model_config_dict['n_heads'],
                        'n_layers': model_config_dict['n_layers'],
                        'dropout': model_config_dict['dropout'],
                        'enc_in': input_features.shape[1],
                        'use_instancenorm': model_config_dict['use_instancenorm'],
                        'n_stores': len(store_encoder.classes_)
                    }
                    
                    model = PatchTST(model_config)
                    model_path = os.path.join(model_info['trial_dir'], 'best.pt')
                    model.load_state_dict(torch.load(model_path, map_location=device))
                    model.to(device)
                    model.eval()
                    
                    # 예측
                    with torch.no_grad():
                        input_tensor = torch.FloatTensor(input_features).unsqueeze(0).to(device)
                        store_id = store_encoder.transform([store])[0]
                        store_id_tensor = torch.LongTensor([store_id]).to(device)
                        
                        pred, _ = model(input_tensor, store_id_tensor)
                        pred = pred.cpu().numpy().flatten()
                        pred = np.clip(pred, 0, None)  # 음수 클리핑
                    
                    model_predictions.append(pred)
                    model_smapes.append(model_info['smape'])
                    
                except Exception as e:
                    print(f"Error in model {model_info['trial_id']}: {str(e)}")
                    continue
            
            if model_predictions:
                # 앙상블 예측
                try:
                    ensemble_pred = ensemble_predictions(model_predictions, config.ensemble_method, model_smapes)
                except:
                    # 백업으로 rank_avg 사용
                    ensemble_pred = ensemble_predictions(model_predictions, 'rank_avg', model_smapes)
                
                store_menu_predictions[store_menu] = ensemble_pred
            else:
                store_menu_predictions[store_menu] = np.zeros(config.pred_len)
        
        all_test_predictions[f'TEST_{test_id:02d}'] = store_menu_predictions
    
    # Top-K 요약 저장
    top_summary = {
        'top_models': top_models,
        'ensemble_method': config.ensemble_method,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(os.path.join(config.best_dir, 'top5_summary.json'), 'w') as f:
        json.dump(top_summary, f, indent=2)
    
    return all_test_predictions

# 12. 제출 파일 생성
def create_submission(test_predictions):
    """제출 파일 생성"""
    if test_predictions is None:
        print("No predictions available, creating zero submission...")
        sample_submission = pd.read_csv(config.submission_path)
        submission = sample_submission.copy()
        # 모든 예측값을 0으로 설정
        for col in submission.columns:
            if col != '구분':
                submission[col] = 0
        
        submission_path = os.path.join(config.result_dir, 'submission_patchtst_optuna_ensemble.csv')
        submission.to_csv(submission_path, index=False)
        print(f"Zero submission file saved: {submission_path}")
        return submission_path
    
    # Sample submission 로드
    sample_submission = pd.read_csv(config.submission_path)
    submission = sample_submission.copy()
    
    # 예측 결과를 제출 형식으로 변환
    for test_name, predictions in test_predictions.items():
        for store_menu, pred in predictions.items():
            if store_menu in submission.columns:
                # TEST_XX 행 찾기
                test_rows = submission[submission['구분'].str.contains(test_name, na=False)]
                if not test_rows.empty:
                    for i, pred_val in enumerate(pred):
                        if i < len(test_rows):
                            row_idx = test_rows.index[i]
                            submission.loc[row_idx, store_menu] = max(0, pred_val)
    
    # 제출 파일 저장
    submission_path = os.path.join(config.result_dir, 'submission_patchtst_optuna_ensemble.csv')
    submission.to_csv(submission_path, index=False)
    print(f"Submission file saved: {submission_path}")
    
    return submission_path

# 13. 메인 실행 함수
def main():
    """메인 실행 함수"""
    print("=" * 60)
    print("PatchTST 시계열 예측 파이프라인 시작")
    print("=" * 60)
    
    try:
        # 1. Optuna 최적화 실행
        print("\n1. Optuna 하이퍼파라미터 최적화 실행...")
        run_optimization()
        
        # 2. 상위 모델 앙상블 예측
        print("\n2. 상위 5개 모델 앙상블 예측...")
        test_predictions = generate_ensemble_predictions()
        
        # 3. 제출 파일 생성
        print("\n3. 제출 파일 생성...")
        submission_path = create_submission(test_predictions)
        
        print("\n" + "=" * 60)
        print("파이프라인 완료!")
        print(f"제출 파일: {submission_path}")
        print(f"Optuna DB: {config.storage_url}")
        print(f"Trial 결과: {config.trials_dir}")
        print(f"Top-5 요약: {config.best_dir}/top5_summary.json")
        print("=" * 60)
        
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

# 14. 결과 분석 및 리포트
def generate_report():
    """결과 분석 리포트 생성"""
    print("\n" + "=" * 50)
    print("결과 분석 리포트")
    print("=" * 50)
    
    # Optuna 결과 분석
    if os.path.exists(config.storage_url.replace('sqlite:///', '')):
        storage = RDBStorage(config.storage_url)
        try:
            study = optuna.load_study(study_name=config.study_name, storage=storage)
            
            print(f"\n[Optuna 최적화 결과]")
            print(f"총 Trial 수: {len(study.trials)}")
            print(f"최적 sMAPE: {study.best_value:.4f}")
            print(f"최적 Trial: {study.best_trial.number}")
            print(f"최적 파라미터:")
            for key, value in study.best_params.items():
                print(f"  {key}: {value}")
                
            # 중요도 분석
            try:
                importance = optuna.importance.get_param_importances(study)
                print(f"\n[파라미터 중요도]")
                for param, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]:
                    print(f"  {param}: {imp:.4f}")
            except:
                pass
                
        except Exception as e:
            print(f"Optuna 결과 로드 실패: {str(e)}")
    
    # Top-K 모델 정보
    top_summary_path = os.path.join(config.best_dir, 'top5_summary.json')
    if os.path.exists(top_summary_path):
        with open(top_summary_path, 'r') as f:
            top_summary = json.load(f)
        
        print(f"\n[Top-{config.top_k} 모델]")
        for i, model in enumerate(top_summary['top_models']):
            print(f"  {i+1}. Trial {model['trial_id']}: sMAPE = {model['smape']:.4f}")
        
        print(f"\n[앙상블 설정]")
        print(f"  방법: {top_summary['ensemble_method']}")
        print(f"  생성 시간: {top_summary['timestamp']}")
    
    # 파일 통계
    print(f"\n[생성된 파일]")
    print(f"  Optuna DB: {config.storage_url}")
    print(f"  Trial 디렉토리: {config.trials_dir} ({len(os.listdir(config.trials_dir)) if os.path.exists(config.trials_dir) else 0}개)")
    print(f"  제출 파일: ./result/submission_patchtst_optuna_ensemble.csv")
    
    print("\n" + "=" * 50)

CUDA available: True
CUDA device count: 4
Current device: 0
Device name: NVIDIA RTX A6000
CUDA version: 12.1
GPU memory: 51.0 GB
Using device: cuda
PyTorch version: 2.4.0+cu121


In [ ]:
# 실행
if __name__ == "__main__":
    main()
    generate_report()

[I 2025-08-19 14:25:34,350] Using an existing study with name 'patchtst_optimization_v2' instead of creating a new one.


PatchTST 시계열 예측 파이프라인 시작

1. Optuna 하이퍼파라미터 최적화 실행...
Completed trials: 0, Remaining: 50
Created new study: patchtst_optimization_v2
